# CIFAR-10 Knowledge Distillation 실습

> **Teacher(큰 CNN)** → **Student(작은 CNN)** 으로 지식을 전달하여, 작은 모델의 정확도를 끌어올리는 과정을 실습합니다.

### 실습 흐름

1. **Teacher 모델 학습** — 큰 CNN(`DeepNN`)을 CIFAR-10으로 학습
2. **Student 단독 학습** — 작은 CNN(`LightNN`)을 혼자 학습 (baseline)
3. **Knowledge Distillation** — Teacher의 Soft Label을 활용해 Student를 학습
4. **3-Way 성능 비교** — Teacher vs Student(KD) vs Student(단독)

### 핵심 원리

일반 학습은 정답(Hard Label)만 보지만, KD는 Teacher의 **확률 분포(Soft Label)** 까지 학습합니다.  
이 Soft Label에는 클래스 간 유사도(**Dark Knowledge**)가 담겨 있어, Student가 더 풍부한 정보를 얻습니다.

## 1. Import & 디바이스 설정

PyTorch 및 torchvision을 로드하고, GPU/MPS/CPU 중 사용 가능한 디바이스를 자동 감지합니다.

In [42]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision.transforms as transforms
import torchvision.datasets as datasets

# GPU(CUDA) > Apple Silicon(MPS) > CPU 순으로 자동 선택
device = torch.accelerator.current_accelerator().type if torch.accelerator.is_available() else "cpu"
print(f"Using {device} device")


Using mps device


## 2. 데이터셋 준비 (CIFAR-10)

**CIFAR-10**: 32×32 컬러 이미지 6만장, 10개 클래스 (비행기, 자동차, 새, 고양이, 사슴, 개, 개구리, 말, 배, 트럭)

- `transforms.Normalize`: ImageNet 통계값으로 정규화 (사전학습 관행)
- 첫 실행 시 `./data/` 폴더에 ~170MB 자동 다운로드

In [43]:
# 이미지 전처리 파이프라인: PIL → Tensor → 정규화(ImageNet 평균/표준편차)
transforms_cifar = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

# 학습용 5만장 + 테스트용 1만장 (첫 실행 시 자동 다운로드)
train_dataset = datasets.CIFAR10(root='./data', train=True, download=True, transform=transforms_cifar)
test_dataset = datasets.CIFAR10(root='./data', train=False, download=True, transform=transforms_cifar)

## 3. DataLoader 생성

`batch_size=128`: 한 번에 128장의 이미지를 묶어서 학습. `num_workers=2`: 데이터 로딩을 2개 프로세스로 병렬화.

In [44]:
# shuffle=True: 매 에폭마다 데이터 순서를 섞어 과적합 방지
train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=128, shuffle=True, num_workers=2)
# shuffle=False: 테스트는 순서 상관없이 일관된 평가
test_loader = torch.utils.data.DataLoader(test_dataset, batch_size=128, shuffle=False, num_workers=2)

## 4. Teacher 모델 정의 (`DeepNN`)

| 구성 | 상세 |
|------|------|
| Conv 블록 | 4층 (128→64→64→32 채널), ReLU, MaxPool |
| FC 블록 | 2048→512→10, Dropout(0.1) |
| 파라미터 | ~1.2M |

복잡한 구조로 높은 정확도를 달성하지만, 크기가 크고 추론이 느립니다.

In [45]:
class DeepNN(nn.Module):
    """Teacher 모델: 4-layer CNN + 2-layer FC"""
    def __init__(self, num_classes=10):
        super(DeepNN, self).__init__()
        self.features = nn.Sequential(
            # 입력: 3채널(RGB) → 128 feature maps
            nn.Conv2d(3, 128, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.Conv2d(128, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),  # 32x32 → 16x16
            nn.Conv2d(64, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.Conv2d(64, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),  # 16x16 → 8x8
        )
        # 32채널 × 8×8 = 2048 → 512 → 10 클래스
        self.classifier = nn.Sequential(
            nn.Linear(2048, 512),
            nn.ReLU(),
            nn.Dropout(0.1),  # 과적합 방지
            nn.Linear(512, num_classes)
        )

    def forward(self, x):
        x = self.features(x)
        x = torch.flatten(x, 1)  # (batch, 32, 8, 8) → (batch, 2048)
        x = self.classifier(x)
        return x  # logits (softmax 전 원시 점수)

## 5. Student 모델 정의 (`LightNN`)

| 구성 | 상세 |
|------|------|
| Conv 블록 | 2층 (16→16 채널), ReLU, MaxPool |
| FC 블록 | 1024→256→10, Dropout(0.1) |
| 파라미터 | ~270K (Teacher의 ~1/4) |

단순한 구조로 빠르지만, 혼자 학습하면 Teacher보다 정확도가 낮습니다.  
**KD의 목표**: 이 Student의 정확도를 Teacher의 지식으로 끌어올리는 것!

In [46]:
class LightNN(nn.Module):
    """Student 모델: 2-layer CNN + 2-layer FC (Teacher의 ~1/4 크기)"""
    def __init__(self, num_classes=10):
        super(LightNN, self).__init__()
        self.features = nn.Sequential(
            # 입력: 3채널(RGB) → 16 feature maps (Teacher는 128)
            nn.Conv2d(3, 16, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),  # 32x32 → 16x16
            nn.Conv2d(16, 16, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),  # 16x16 → 8x8
        )
        # 16채널 × 8×8 = 1024 → 256 → 10 클래스
        self.classifier = nn.Sequential(
            nn.Linear(1024, 256),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(256, num_classes)
        )

    def forward(self, x):
        x = self.features(x)
        x = torch.flatten(x, 1)  # (batch, 16, 8, 8) → (batch, 1024)
        x = self.classifier(x)
        return x  # logits


## 6. 일반 학습 함수 (Hard Label)

정답 레이블만 사용하는 표준 학습 방식입니다.  
`CrossEntropyLoss`: 모델 출력과 정답(one-hot) 사이의 차이를 측정하여 역전파합니다.

In [47]:
def train(model, train_loader, epochs, learning_rate, device):
    criterion = nn.CrossEntropyLoss()  # Hard Label Loss: 정답 클래스만 1, 나머지 0
    optimizer = optim.Adam(model.parameters(), lr=learning_rate)

    model.train()  # 학습 모드 (Dropout 활성화)

    for epoch in range(epochs):
        running_loss = 0.0
        for inputs, labels in train_loader:
            inputs, labels = inputs.to(device), labels.to(device)

            optimizer.zero_grad()       # 이전 기울기 초기화
            outputs = model(inputs)     # 순전파: 이미지 → logits (10개 클래스 점수)
            loss = criterion(outputs, labels)  # 정답과의 차이 계산
            loss.backward()             # 역전파: 기울기 계산
            optimizer.step()            # 가중치 업데이트

            running_loss += loss.item()

        print(f"Epoch {epoch+1}/{epochs}, Loss: {running_loss / len(train_loader)}")


## 7. 평가 함수

테스트셋(1만장)에 대해 정확도를 측정합니다. `torch.no_grad()`로 기울기 계산을 비활성화하여 메모리를 절약합니다.

In [48]:
def test(model, test_loader, device):
    model.to(device)
    model.eval()  # 평가 모드 (Dropout 비활성화, BatchNorm 고정)

    correct = 0
    total = 0

    with torch.no_grad():  # 평가 시에는 기울기 계산 불필요 → 메모리 절약
        for inputs, labels in test_loader:
            inputs, labels = inputs.to(device), labels.to(device)

            outputs = model(inputs)                      # logits: (batch, 10)
            _, predicted = torch.max(outputs.data, 1)    # 가장 높은 점수의 클래스 인덱스

            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    accuracy = 100 * correct / total
    print(f"Test Accuracy: {accuracy:.2f}%")
    return accuracy


## 8. Teacher 학습

Teacher(`DeepNN`)를 10 에폭 동안 학습시킵니다. 이 모델이 충분히 높은 정확도를 가져야 Student에게 전달할 "지식"이 의미가 있습니다.

> `manual_seed(42)`: 재현 가능한 실험을 위해 난수 시드를 고정합니다.

In [ ]:
print("🎓 Step 1: Teacher(DeepNN) 훈련 중...")
torch.manual_seed(42)  # 재현성을 위한 시드 고정
nn_deep = DeepNN(num_classes=10).to(device)
train(nn_deep, train_loader, epochs=10, learning_rate=0.001, device=device)
test_accuracy_deep = test(nn_deep, test_loader, device)


🎓 Step 1: Teacher(DeepNN) 훈련 중...
Epoch 1/20, Loss: 1.359011491088916
Epoch 2/20, Loss: 0.890776449457154
Epoch 3/20, Loss: 0.6943295636140477
Epoch 4/20, Loss: 0.553986884139078
Epoch 5/20, Loss: 0.4455798629604642
Epoch 6/20, Loss: 0.3384586377903019
Epoch 7/20, Loss: 0.24859476920283968
Epoch 8/20, Loss: 0.18707121472300775
Epoch 9/20, Loss: 0.15094050651659136
Epoch 10/20, Loss: 0.13059403941683148
Epoch 11/20, Loss: 0.10863088007987765
Epoch 12/20, Loss: 0.1028550926815061
Epoch 13/20, Loss: 0.09417858298705972
Epoch 14/20, Loss: 0.08460053711024391
Epoch 15/20, Loss: 0.08344356393646402
Epoch 16/20, Loss: 0.07781012578750662
Epoch 17/20, Loss: 0.0691573049461045
Epoch 18/20, Loss: 0.07835882343232746
Epoch 19/20, Loss: 0.07065128292078557
Epoch 20/20, Loss: 0.0718422512049832
Test Accuracy: 75.32%


## 9. Student 모델 2개 생성 (공정 비교용)

같은 시드(`manual_seed(42)`)로 Student를 2개 만들어, **동일한 초기 가중치**에서 출발하도록 합니다.

| 모델 | 학습 방식 | 목적 |
|------|----------|------|
| `nn_light` | Hard Label만 (일반 학습) | baseline |
| `new_nn_light` | **KD** (Teacher의 Soft Label 활용) | 증류 효과 검증 |

> 초기 가중치가 동일해야 "학습 방식의 차이"만 비교할 수 있습니다.

In [50]:
# Student A: 혼자 공부 (baseline, Hard Label만)
torch.manual_seed(42)
nn_light = LightNN(num_classes=10).to(device)

# Student B: KD로 공부 (Teacher의 Soft Label 활용)
torch.manual_seed(42)
new_nn_light = LightNN(num_classes=10).to(device)

# 두 모델의 초기 가중치가 동일한지 확인 (norm 비교)
print("Norm of 1st layer of nn_light:", torch.norm(nn_light.features[0].weight).item())
print("Norm of 1st layer of new_nn_light:", torch.norm(new_nn_light.features[0].weight).item())


Norm of 1st layer of nn_light: 2.327361822128296
Norm of 1st layer of new_nn_light: 2.327361822128296


## 10. 파라미터 수 비교

Teacher와 Student의 모델 크기 차이를 확인합니다.  
파라미터 수가 적을수록 추론이 빠르고 메모리를 적게 사용하지만, 일반적으로 정확도는 낮아집니다.  
**KD는 이 trade-off를 완화**하는 기법입니다.

In [51]:
# 모든 레이어의 파라미터 수를 합산
total_params_deep = "{:,}".format(sum(p.numel() for p in nn_deep.parameters()))
print(f"DeepNN (Teacher) parameters: {total_params_deep}")

total_params_light = "{:,}".format(sum(p.numel() for p in nn_light.parameters()))
print(f"LightNN (Student) parameters: {total_params_light}")


DeepNN (Teacher) parameters: 1,186,986
LightNN (Student) parameters: 267,738


## 11. Student 단독 학습 (Baseline)

`nn_light`를 **Hard Label만으로** 10 에폭 학습합니다.  
이 결과가 KD의 효과를 비교할 기준선(baseline)이 됩니다.

In [52]:
print("📚 Step 2: Student(LightNN) 단독 학습 중... (Hard Label only)")
train(nn_light, train_loader, epochs=10, learning_rate=0.001, device=device)
test_accuracy_light_ce = test(nn_light, test_loader, device)


📚 Step 2: Student(LightNN) 단독 학습 중... (Hard Label only)
Epoch 1/10, Loss: 1.4599939624366858
Epoch 2/10, Loss: 1.153122827220146
Epoch 3/10, Loss: 1.0206042615044149
Epoch 4/10, Loss: 0.9208040501150634
Epoch 5/10, Loss: 0.8456781596478904
Epoch 6/10, Loss: 0.7813222766532313
Epoch 7/10, Loss: 0.719012732548482
Epoch 8/10, Loss: 0.6568071686703226
Epoch 9/10, Loss: 0.6014061779774669
Epoch 10/10, Loss: 0.5539262728465487
Test Accuracy: 70.57%


## 12. 중간 결과 비교 & KD가 필요한 이유

Teacher와 Student(단독)의 정확도를 비교하고, 왜 Soft Label(Dark Knowledge)이 유용한지 설명합니다.

### Hard Label vs Soft Label

| 방식 | 트럭 사진에 대한 레이블 | 정보량 |
|------|----------------------|--------|
| **Hard** | `[0,0,0,0,0,0,0,0,0,1]` — "트럭!" | 정답만 |
| **Soft** | `[0.04,0.15,0.00,0.00,...,0.80]` — "트럭이지만 자동차와 비슷" | 클래스 간 유사도 포함 |

Soft Label의 작은 확률값들에 담긴 정보 = **Dark Knowledge** (Hinton, 2015)

In [53]:
# 중간 성능 비교: Teacher vs Student(단독)
print(f"Teacher accuracy:  {test_accuracy_deep:.2f}%")
print(f"Student accuracy:  {test_accuracy_light_ce:.2f}%")
print(f"Gap:               {test_accuracy_deep - test_accuracy_light_ce:.2f}%p")

print("\n" + "="*50)
print("왜 Soft Label(Dark Knowledge)이 유용한가?")
print("="*50)

# Hard Label의 한계:
# - 정답 클래스만 1, 나머지 0 → 클래스 간 관계 정보가 완전히 손실됨
# - "트럭 ≈ 자동차" 같은 유사성을 학습할 수 없음

# Soft Label의 장점:
# - Teacher가 출력하는 확률 분포에는 클래스 간 관계가 포함됨
# - 예: 트럭 이미지 → Teacher 출력: [트럭:0.80, 자동차:0.15, 비행기:0.04, ...]
# - Student는 "트럭과 자동차가 비슷하다"는 것까지 학습 가능

print("""
Hard Label: "이건 트럭이야!" (정답만, 관계 정보 없음)
Soft Label: "트럭 80%, 자동차 15%, 비행기 4%..." (클래스 간 유사도 포함)

→ Soft Label의 작은 확률값들이 바로 'Dark Knowledge'입니다.
  Student가 이를 학습하면, 단순 암기를 넘어 개념 이해에 가까운 학습이 가능합니다.
""")


Teacher accuracy:  75.32%
Student accuracy:  70.57%
Gap:               4.75%p

왜 Soft Label(Dark Knowledge)이 유용한가?

Hard Label: "이건 트럭이야!" (정답만, 관계 정보 없음)
Soft Label: "트럭 80%, 자동차 15%, 비행기 4%..." (클래스 간 유사도 포함)

→ Soft Label의 작은 확률값들이 바로 'Dark Knowledge'입니다.
  Student가 이를 학습하면, 단순 암기를 넘어 개념 이해에 가까운 학습이 가능합니다.



## 13. Knowledge Distillation 실행

이제 핵심인 KD 학습을 수행합니다. Student는 두 가지 Loss를 동시에 최소화합니다:

$$L_{total} = 0.25 \times T^2 \times KL\bigl(\sigma(z_t/T),\ \sigma(z_s/T)\bigr) + 0.75 \times CE(y,\ \hat{y}_s)$$

| 하이퍼파라미터 | 값 | 역할 |
|---------------|-----|------|
| **Temperature (T)** | 2 | logits를 T로 나눠 확률 분포를 부드럽게 → Dark Knowledge 노출 |
| **soft_target_loss_weight** | 0.25 | Teacher Soft Label 모방 비중 |
| **ce_loss_weight** | 0.75 | 실제 정답 학습 비중 |

> **T² 보정**: Temperature로 나누면 gradient가 1/T² 만큼 줄어드므로, T²을 곱해 보정합니다 (Hinton, 2015).

In [ ]:
def train_knowledge_distillation(teacher, student, train_loader, epochs, learning_rate, T, soft_target_loss_weight, ce_loss_weight, device):
    """
    Knowledge Distillation 학습 함수

    Student가 두 가지를 동시에 학습:
      1. 실제 정답 레이블 (CE Loss)
      2. Teacher의 softened 확률 분포 (KL Divergence)

    Args:
        T: Temperature — 높을수록 확률 분포가 부드러워져 Dark Knowledge가 더 많이 노출됨
        soft_target_loss_weight: KL Loss 가중치 (Teacher 지식 비중)
        ce_loss_weight: CE Loss 가중치 (정답 레이블 비중)
    """
    ce_loss = nn.CrossEntropyLoss()
    optimizer = optim.Adam(student.parameters(), lr=learning_rate)

    teacher.eval()   # Teacher는 학습하지 않음 (가중치 고정)
    student.train()  # Student만 학습

    for epoch in range(epochs):
        running_loss = 0.0
        for inputs, labels in train_loader:
            inputs, labels = inputs.to(device), labels.to(device)

            optimizer.zero_grad()

            # Step 1: Teacher의 예측 (gradient 계산 불필요)
            with torch.no_grad():
                teacher_logits = teacher(inputs)

            # Step 2: Student의 예측
            student_logits = student(inputs)

            # Step 3: Temperature Scaling
            # T로 나누면 분포가 부드러워짐 → 작은 확률값(Dark Knowledge)이 더 잘 보임
            # T=1: [0.95, 0.03, 0.02]  →  T=3: [0.56, 0.24, 0.20]
            soft_targets = nn.functional.softmax(teacher_logits / T, dim=-1)
            soft_prob = nn.functional.log_softmax(student_logits / T, dim=-1)

            # Step 4: KL Divergence (Teacher ∥ Student)
            # Teacher와 Student의 확률 분포가 얼마나 다른지 측정
            # T²을 곱하는 이유: Temperature scaling으로 줄어든 gradient를 보정 (Hinton, 2015)
            soft_targets_loss = torch.sum(soft_targets * (soft_targets.log() - soft_prob)) / soft_prob.size()[0] * (T**2)

            # Step 5: CE Loss (정답 레이블)
            label_loss = ce_loss(student_logits, labels)

            # Step 6: 최종 Loss = KL 가중합 + CE 가중합
            loss = soft_target_loss_weight * soft_targets_loss + ce_loss_weight * label_loss

            loss.backward()   # 역전파 (Student 가중치에 대해서만)
            optimizer.step()  # Student 가중치 업데이트

            running_loss += loss.item()

        print(f"Epoch {epoch+1}/{epochs}, Loss: {running_loss / len(train_loader)}")


# Student(KD)를 다시 초기화 (이전 학습 결과 제거)
torch.manual_seed(42)
new_nn_light = LightNN(num_classes=10).to(device)

# KD 학습 실행
print("🎓 Step 3: Knowledge Distillation으로 Student 훈련 중...")
print("   설정: T=2, KL_weight=0.25, CE_weight=0.75")

train_knowledge_distillation(
    teacher=nn_deep,
    student=new_nn_light,
    train_loader=train_loader,
    epochs=10,
    learning_rate=0.001,
    T=2,                         # Temperature: 확률 분포를 부드럽게 (T²=4)
    soft_target_loss_weight=0.25,  # Teacher 지식 25%
    ce_loss_weight=0.75,           # 정답 레이블 75%
    device=device
)

test_accuracy_light_ce_and_kd = test(new_nn_light, test_loader, device)

# ═══════════════════════════════════════════════════
# 최종 3-Way 비교
# ═══════════════════════════════════════════════════
print("\n" + "="*50)
print("🏆 최종 성능 비교")
print("="*50)

print(f"  Teacher (DeepNN):          {test_accuracy_deep:.2f}%")
print(f"  Student 단독 (Hard Label): {test_accuracy_light_ce:.2f}%")
print(f"  Student KD (Soft Label):   {test_accuracy_light_ce_and_kd:.2f}%")
print(f"\n  KD 효과: {test_accuracy_light_ce_and_kd - test_accuracy_light_ce:.2f}%p 향상")

# KD Student가 단독 Student보다 높은 정확도를 보이면 증류 성공!
# 같은 크기의 모델이지만, Teacher의 Dark Knowledge를 활용해 더 나은 성능을 달성합니다.


🎓 Step 3: Knowledge Distillation으로 Student 훈련 중...
   설정: T=4, KL_weight=0.5, CE_weight=0.5
Epoch 1/10, Loss: 9.952079104645478
Epoch 2/10, Loss: 7.694389551801755
Epoch 3/10, Loss: 6.76645168441031
Epoch 4/10, Loss: 6.051788226417873
Epoch 5/10, Loss: 5.529599786109632
Epoch 6/10, Loss: 5.106339121718541
Epoch 7/10, Loss: 4.734021183779783
Epoch 8/10, Loss: 4.369742049585523
Epoch 9/10, Loss: 4.0809470759633255
Epoch 10/10, Loss: 3.8221321307179874
Test Accuracy: 70.40%

🏆 최종 성능 비교
  Teacher (DeepNN):          75.32%
  Student 단독 (Hard Label): 70.57%
  Student KD (Soft Label):   70.40%

  KD 효과: -0.17%p 향상
